# 02 - UNet Segmentation

Arquitetura UNet do notebook Kaggle `skin-cancer-lesions-segmentation-using-unet.ipynb`,
portada para PyTorch e adaptada para segmentacao binaria de lesoes de pele.

O modelo e salvo em `outputs/models/unet_segmentation.pt` e usado em
`03_preprocessing.ipynb` para predizer mascaras durante o pre-processamento.


## 1. Imports e configuracao

In [24]:
import glob
import random
from pathlib import Path

import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

print(f"torch version : {torch.__version__}")
print(f"CUDA          : {torch.cuda.is_available()}")
print(f"MPS           : {torch.backends.mps.is_available()}")


def resolve_repo_root() -> Path:
    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
    for candidate in candidates:
        if (candidate / "data" / "metadata.csv").exists() and (candidate / "docs").exists():
            return candidate
    raise FileNotFoundError("Raiz do repositorio nao encontrada.")


ROOT_DIR   = resolve_repo_root()
IMAGES_DIR = ROOT_DIR / "data" / "images"
MASKS_DIR  = ROOT_DIR / "data" / "masks"
MODEL_DIR  = ROOT_DIR / "outputs" / "models"
FIG_DIR    = ROOT_DIR / "outputs" / "figures"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE   = 64
BATCH_SIZE = 32
EPOCHS     = 20
LR         = 1e-3
SEED       = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print(f"Device: {DEVICE}")


torch version : 2.11.0
CUDA          : False
MPS           : True
Device: mps


## 2. Carregamento e split dos dados

In [25]:
image_files = sorted(glob.glob(str(IMAGES_DIR / "*.jpg")))
mask_files  = sorted(glob.glob(str(MASKS_DIR  / "*.png")))

img_stem  = {Path(f).stem: f for f in image_files}
mask_stem = {Path(f).stem: f for f in mask_files}

common = sorted(set(img_stem) & set(mask_stem))
image_paths = [img_stem[k]  for k in common]
mask_paths  = [mask_stem[k] for k in common]

train_images, test_images, train_masks, test_masks = train_test_split(
    image_paths, mask_paths, test_size=0.2, random_state=SEED
)
train_images, val_images, train_masks, val_masks = train_test_split(
    train_images, train_masks, test_size=0.1, random_state=SEED
)

print(f"Train: {len(train_images)}  Val: {len(val_images)}  Test: {len(test_images)}")


Train: 7210  Val: 802  Test: 2003


## 3. Dataset e DataLoaders

In [26]:
class SkinLesionDataset(Dataset):
    def __init__(self, image_paths, mask_paths):
        self.image_paths = image_paths
        self.mask_paths  = mask_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = np.array(Image.open(self.image_paths[idx]).convert("RGB"))
        image = cv2.resize(image, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        image = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0

        mask = np.array(Image.open(self.mask_paths[idx]).convert("L"))
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        mask = torch.from_numpy((mask > 127).astype(np.float32)).unsqueeze(0)
        return image, mask


train_ds = SkinLesionDataset(train_images, train_masks)
val_ds   = SkinLesionDataset(val_images,   val_masks)
test_ds  = SkinLesionDataset(test_images,  test_masks)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Batches — train: {len(train_loader)}  val: {len(val_loader)}  test: {len(test_loader)}")


Batches — train: 226  val: 26  test: 63


## 4. Visualizacao de amostras

In [27]:
images, masks = next(iter(train_loader))
fig, axes = plt.subplots(3, 2, figsize=(6, 9))
for row in range(3):
    axes[row, 0].imshow(images[row].permute(1, 2, 0).numpy())
    axes[row, 0].set_title("Imagem"); axes[row, 0].axis("off")
    axes[row, 1].imshow(masks[row].squeeze().numpy(), cmap="gray")
    axes[row, 1].set_title("Mascara GT"); axes[row, 1].axis("off")
plt.tight_layout()
plt.savefig(str(FIG_DIR / "unet_dataset_samples.png"), dpi=100)
plt.show()


/var/folders/rt/cjr17y490k14_vcjyjsbc73w0000gn/T/ipykernel_28018/4059888680.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Arquitetura UNet

Porta fiel da arquitetura Kaggle (`double_conv_block`, `downsample_block`, `upsample_block`)
para PyTorch. Saida: 1 canal com sigmoid para segmentacao binaria.


In [28]:
def double_conv_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
    )


class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.drop = nn.Dropout2d(0.3)
        self.pool = nn.MaxPool2d(2)

        # encoder (downsample blocks)
        self.enc1 = double_conv_block(3,   64)
        self.enc2 = double_conv_block(64,  128)
        self.enc3 = double_conv_block(128, 256)
        self.enc4 = double_conv_block(256, 512)

        # bridge
        self.bridge = double_conv_block(512, 1024)

        # decoder (upsample blocks)
        self.up4  = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = double_conv_block(1024, 512)
        self.drop_dec4 = nn.Dropout2d(0.25)

        self.up3  = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = double_conv_block(512, 256)
        self.drop_dec3 = nn.Dropout2d(0.25)

        self.up2  = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = double_conv_block(256, 128)
        self.drop_dec2 = nn.Dropout2d(0.25)

        self.up1  = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = double_conv_block(128, 64)
        self.drop_dec1 = nn.Dropout2d(0.25)

        self.out_conv = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, x):
        # encoder
        s1 = self.enc1(x)
        s2 = self.enc2(self.drop(self.pool(s1)))
        s3 = self.enc3(self.drop(self.pool(s2)))
        s4 = self.enc4(self.drop(self.pool(s3)))

        # bridge
        b = self.bridge(self.drop(self.pool(s4)))

        # decoder
        x = self.drop_dec4(torch.cat([self.up4(b),  s4], dim=1)); x = self.dec4(x)
        x = self.drop_dec3(torch.cat([self.up3(x),  s3], dim=1)); x = self.dec3(x)
        x = self.drop_dec2(torch.cat([self.up2(x),  s2], dim=1)); x = self.dec2(x)
        x = self.drop_dec1(torch.cat([self.up1(x),  s1], dim=1)); x = self.dec1(x)

        return self.out_conv(x)


model = UNet().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parametros: {n_params:,}")


Parametros: 31,031,745


## 6. Treinamento

In [29]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history = {"train_loss": [], "val_loss": []}
best_val_loss = float("inf")
best_state    = None
patience      = 4
patience_ctr  = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for images, masks in train_loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(images), masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(images)
    train_loss /= len(train_ds)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            val_loss += criterion(model(images), masks).item() * len(images)
    val_loss /= len(val_ds)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    improved = val_loss < best_val_loss
    print(f"Epoch {epoch:02d}/{EPOCHS} | train={train_loss:.4f} val={val_loss:.4f}"
          + (" <<<" if improved else ""))

    if improved:
        best_val_loss = val_loss
        best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_ctr  = 0
    else:
        patience_ctr += 1
        if patience_ctr >= patience:
            print(f"Early stopping na epoca {epoch}.")
            break

model.load_state_dict(best_state)


Epoch 01/20 | train=0.3587 val=0.2039 <<<
Epoch 02/20 | train=0.1884 val=0.1412 <<<
Epoch 03/20 | train=0.1486 val=0.1180 <<<
Epoch 04/20 | train=0.1401 val=0.1100 <<<
Epoch 05/20 | train=0.1300 val=0.1208
Epoch 06/20 | train=0.1259 val=0.1135
Epoch 07/20 | train=0.1218 val=0.1014 <<<
Epoch 08/20 | train=0.1205 val=0.1211
Epoch 09/20 | train=0.1180 val=0.0998 <<<
Epoch 10/20 | train=0.1132 val=0.1066
Epoch 11/20 | train=0.1099 val=0.1043
Epoch 12/20 | train=0.1083 val=0.0985 <<<
Epoch 13/20 | train=0.1125 val=0.1392
Epoch 14/20 | train=0.1130 val=0.0934 <<<
Epoch 15/20 | train=0.1082 val=0.1000
Epoch 16/20 | train=0.1054 val=0.1007
Epoch 17/20 | train=0.1020 val=0.0957
Epoch 18/20 | train=0.0998 val=0.0969
Early stopping na epoca 18.


<All keys matched successfully>

## 7. Curvas de treinamento

In [30]:
plt.figure(figsize=(8, 4))
plt.plot(history["train_loss"], "r", label="train loss")
plt.plot(history["val_loss"],   "b", label="val loss")
plt.xlabel("Epoch"); plt.ylabel("BCE Loss")
plt.title("Loss Graph"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(str(FIG_DIR / "unet_learning_curves.png"), dpi=100)
plt.show()


/var/folders/rt/cjr17y490k14_vcjyjsbc73w0000gn/T/ipykernel_28018/3705233237.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Metricas no test set

In [31]:
model.eval()
tp = fp = fn = tn = 0
with torch.no_grad():
    for images, masks in test_loader:
        preds = (torch.sigmoid(model(images.to(DEVICE))) > 0.5).float().cpu()
        tp += (preds * masks).sum().item()
        fp += (preds * (1 - masks)).sum().item()
        fn += ((1 - preds) * masks).sum().item()
        tn += ((1 - preds) * (1 - masks)).sum().item()

iou  = tp / (tp + fp + fn + 1e-8)
dice = 2 * tp / (2 * tp + fp + fn + 1e-8)
acc  = (tp + tn) / (tp + fp + fn + tn + 1e-8)
print(f"IoU: {iou:.4f} | Dice: {dice:.4f} | Pixel Acc: {acc:.4f}")


IoU: 0.8589 | Dice: 0.9241 | Pixel Acc: 0.9588


In [32]:
fig, axes = plt.subplots(4, 3, figsize=(9, 12))
images, masks = next(iter(test_loader))
with torch.no_grad():
    preds = torch.sigmoid(model(images.to(DEVICE))).cpu()

for row in range(4):
    axes[row, 0].imshow(images[row].permute(1, 2, 0).numpy())
    axes[row, 0].set_title("Imagem"); axes[row, 0].axis("off")
    axes[row, 1].imshow(masks[row].squeeze().numpy(), cmap="gray")
    axes[row, 1].set_title("Mascara GT"); axes[row, 1].axis("off")
    axes[row, 2].imshow((preds[row].squeeze().numpy() > 0.5).astype(float), cmap="gray")
    axes[row, 2].set_title("Mascara Predita"); axes[row, 2].axis("off")

plt.tight_layout()
plt.savefig(str(FIG_DIR / "unet_segmentation_predictions.png"), dpi=100)
plt.show()


/var/folders/rt/cjr17y490k14_vcjyjsbc73w0000gn/T/ipykernel_28018/1044803897.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Salvar modelo

In [33]:
MODEL_PATH = MODEL_DIR / "unet_segmentation.pt"
torch.save(best_state, MODEL_PATH)
print(f"Modelo salvo em: {MODEL_PATH}")


Modelo salvo em: /Users/eduardoyaginuma/Documents/Repositorios/insper/skin-cancer-images-segmentation/outputs/models/unet_segmentation.pt
